In [1]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

g:\Genarative-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
docs = (
    PyPDFLoader('.\Document\Company_Policies.pdf').load() +
    PyPDFLoader('.\Document\Company_Profile.pdf').load() +
    PyPDFLoader('.\Document\Product_and_Pricing.pdf').load()
    
)

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
C:\Users\hmaru\AppData\Local\Temp\ipykernel_5348\2514151863.py:2: SyntaxWarning: invalid escape sequence '\D'
  PyPDFLoader('.\Document\Company_Policies.pdf').load() +
C:\Users\hmaru\AppData\Local\Temp\ipykernel_5348\2514151863.py:3: SyntaxWarning: invalid escape sequence '\D'
  PyPDFLoader('.\Document\Company_Profile.pdf').load() +
C:\Users\hmaru\AppData\Local\Temp\ipykernel_5348\2514151863.py:4: SyntaxWarning: invalid escape sequence '\D'
  PyPDFLoader('.\Document\Product_and_Pricing.pdf').load()
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


In [4]:
chunks = RecursiveCharacterTextSplitter(chunk_size = 600,chunk_overlap = 150).split_documents(docs)
print(len(chunks))

16


In [5]:
embed_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2364.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
vector_store = FAISS.from_documents(chunks,embed_model)
retriever = vector_store.as_retriever(search_type = 'similarity',search_kwargs={'k':10})

In [7]:
import os
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [120]:
class State(TypedDict):
    question:str
    need_retrieval:bool

    docs:list[Document]
    relevanced_docs:list[Document]
    context:str

    issup:Literal["fully_supported","partially_supported","no_support"]
    evidence:list[str]
    out:list
    answer:str

In [121]:
from langchain_core.output_parsers import StrOutputParser
class RetrieveDecision(BaseModel):
    should_retrieve:bool = Field(
        ...,
        description = "True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}")
    ]
)

retrieve_decied_chain = decide_retrieval_prompt | llm.with_structured_output(
    RetrieveDecision,
    method="json_mode"  
)
def decide_revrieval_node(state: State):
    decision:RetrieveDecision = retrieve_decied_chain.invoke({'question':state['question']})

    return {'need_retrieval':decision.should_retrieve}

In [122]:
class Isrelevant(BaseModel):
    is_relevant:bool = Field(
        ...,
        description='True if the document helps answer the question, else False.'
    )

Isrelevant_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',
        "You are judging document relevance.\n"
            "Return JSON that matches this schema:\n"
            "{{'is_relevant': boolean}}\n\n"
            "A document is relevant if it contains information useful for answering the question."
        ),
        ('human','\nQuestion: {question}\n\nDocument:\n{document}')
    ]
)

isrelevant_chain = Isrelevant_prompt | llm.with_structured_output(Isrelevant,method='json_mode') 

def isrelevant_node(state:State):
    docs = state['docs']
    qus = state['question']
    all_docs = []
    all_docs_input = []
    relevanced_doc:list[Document] = []

    for d  in docs:
       
        all_docs_input.append({'question':qus,'document':d.page_content})
    
    all_docs_out = isrelevant_chain.batch(all_docs_input)

    for d,r in zip(docs,all_docs_out):
        if r.is_relevant:
            relevanced_doc.append(d)

    return {
        'relevanced_docs': relevanced_doc,
        'out':all_docs_out
        
        }
        


In [123]:
generation_prompt = ChatPromptTemplate.from_messages([
     (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", '\nQuestion: {question}')]
        
)

genaration_chain = generation_prompt | llm | StrOutputParser()

def generation_node(state:State):
    out = genaration_chain.invoke({'question':state['question']})
    return {'answer': out, 'context': ''}

In [124]:
main_generation_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',
        "You are a business RAG assistant.\n"
        "Answer the user's question using ONLY the provided context.\n"
        "If the context does not contain enough information, say:\n"
        "'No relevant document found.'\n"
        "Do not use outside knowledge.\n"
        ),
        ('human','\nquestion:{question}\n\ndocument:{context}')
    ]
)

main_generation_chain = main_generation_prompt | llm | StrOutputParser()

def main_generation_node(state:State):
    qus = state['question']
    docs = state.get('relevanced_docs',[])

    context = "\n\n---\n\n".join(d.page_content for d in docs).strip()
    if not context:
        return{'answer':"I dont Know,No relevant document found"}
        
    ans = main_generation_chain.invoke({'question':qus,'context':context})
    return {
        'answer':ans,
        'context':context
        }

In [125]:
def no_relevant_docs(state:State):
    return{'answer':"I dont Know,No relevant document found"}


In [126]:
class IsSupDecision(BaseModel):
    issup: Literal["fully_supported","partially_supported","no_support"]
    evidence: list[str] = Field(default_factory=list)

issup_prompt = ChatPromptTemplate([
    ('system',
    "You are verifying whether the ANSWER is supported by the CONTEXT.\n"
            "Return JSON with keys: issup, evidence.\n"
            "issup must be one of: fully_supported, partially_supported, no_support.\n\n"
            "How to decide issup:\n"
            "- fully_supported:\n"
            "  Every meaningful claim is explicitly supported by CONTEXT, and the ANSWER does NOT introduce\n"
            "  any qualitative/interpretive words that are not present in CONTEXT.\n"
            "  (Examples of disallowed words unless present in CONTEXT: culture, generous, robust, designed to,\n"
            "  supports professional development, best-in-class, employee-first, etc.)\n\n"
            "- partially_supported:\n"
            "  The core facts are supported, BUT the ANSWER includes ANY abstraction, interpretation, or qualitative\n"
            "  phrasing not explicitly stated in CONTEXT (e.g., calling policies 'culture', saying leave is 'generous',\n"
            "  or inferring outcomes like 'supports professional development').\n\n"
            "- no_support:\n"
            "  The key claims are not supported by CONTEXT.\n\n"
            "Rules:\n"
            "- Be strict: if you see ANY unsupported qualitative/interpretive phrasing, choose partially_supported.\n"
            "- If the answer is mostly unrelated to the question or unsupported, choose no_support.\n"
            "- Evidence: list up to 3 short direct quotes from CONTEXT as plain strings only. Each evidence item MUST be a string, not an object or dict.\n"
            "- Do not use outside knowledge."
            ),
    (
            "human",
            "Question:\n{question}\n\n"
            "Answer:\n{answer}\n\n"
            "Context:\n{context}\n"
        )
])

issup_chain = issup_prompt | llm.with_structured_output(IsSupDecision,method='json_mode')

def issup_node(state:State):
    qus =  state['question']
    answer = state['answer']
    context = state.get('context', '')  # safe get

    if not context:
        return {
            'issup': 'no_support',
            'evidence': []
        }

    decision:IsSupDecision = issup_chain.invoke({'question':qus,'answer':answer,'context':context})
    return {
        'issup':decision.issup,
        'evidence':decision.evidence
    }

In [127]:
def route_after_relevance(state:State) ->Literal['main_generation_chain','no_relevant_docs']:
    if state.get('relevanced_docs') and len(state.get('relevanced_docs')) > 0:
        return "main_generation_node"
    return "no_relevant_docs"


In [128]:
def retrieve_node(state:State):
    return {'docs':retriever.invoke(state['question'])}

In [129]:
def route_after_decide(state:State) -> Literal['generate_direct','retrieve']:
    if state['need_retrieval']:
        return 'retrieve'
    return 'generate_direct'


In [130]:
g = StateGraph(State)

# --------------------
# Nodes
# --------------------
g.add_node("decide_retrieval", decide_revrieval_node)
g.add_node("generate_direct", generation_node)
g.add_node("retrieve", retrieve_node)
g.add_node('isrelevanced',isrelevant_node)
g.add_node('main_generation_node',main_generation_node)
g.add_node("issup_node",issup_node)

# --------------------
# Edges
# --------------------
g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)
g.add_edge('retrieve','isrelevanced')
g.add_edge('isrelevanced','main_generation_node')
g.add_edge("main_generation_node",'issup_node') 
g.add_edge('issup_node',END) 


app = g.compile()


In [133]:
result = app.invoke(
    {
        "question": "do nexaAI include a free trail? if yes,how many days",
        "need_retrieval": False,
        "docs": [],

        "answer": "",
    }
)

print(result['answer'])




RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01jpsat1tje228zdsd9wvsmfwe` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5848, Requested 369. Please try again in 2.17s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [114]:

print(len(result['docs']))
r = 'issup'
result[r]

10


'partially_supported'